IT DOESN'T WORK

In [8]:
# ---------- apple_financials_2001_2009_annuals_quarters.py ----------
import re, time, requests, pandas as pd
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from datetime import datetime, date
from io import StringIO  # <-- for read_html(StringIO(html))

# -------- CONFIG --------
USER_AGENT = "Jon jonmcgowan15@gmail.com"
HEADERS = {"User-Agent": USER_AGENT}
PAUSE = 0.35
CIK = "0000320193"
FORMS = {"10-K","10-Q"}

DATE_FROM = "2001-01-01"
DATE_TO   = "2009-12-31"

LABELS_HTML = {
    "revenue": [r"^net\s+sales$", r"^net\s+revenue[s]?$", r"^sales$", r"^revenues?$"],
    "net_income": [r"^net\s+(income|loss)$"],
    "assets": [r"^total\s+assets$"],
    "liabilities": [r"^total\s+liabilities$"],
    "equity": [r"^total\s+(share|stock)holders[’']?\s*equity$"]
}

LABELS_TEXT = {
    "revenue": [r"net sales[^$:\n]*[\$ ]([\d,\.]+)"],
    "net_income": [r"net income[^$:\n]*[\$ ]([\d,\.]+)"],
    "assets": [r"total assets[^$:\n]*[\$ ]([\d,\.]+)"],
    "liabilities": [r"total liabilities[^$:\n]*[\$ ]([\d,\.]+)"],
    "equity": [r"(?:shareholders'|stockholders') equity[^$:\n]*[\$ ]([\d,\.]+)"]
}

YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")
DATE_PATTERNS = [
    "%B %d, %Y", "%b %d, %Y",
    "%m/%d/%Y", "%m/%d/%y",
    "%d %B %Y", "%d %b %Y",
]

# ------------------- HTTP helpers -------------------
def _get(url, timeout=60, tries=4, backoff=0.9):
    last_err=None
    for k in range(tries):
        try:
            r=requests.get(url,headers=HEADERS,timeout=timeout)
            if r.status_code in (403,429):
                time.sleep(backoff*(k+1))
                continue
            r.raise_for_status()
            return r
        except Exception as e:
            last_err=e
            time.sleep(backoff*(k+1))
    raise last_err

def fetch_json(url): return _get(url).json()
def fetch_text(url): return _get(url).text

# ------------------- Filings -------------------
def get_filings_json(cik):
    sub_url=f"https://data.sec.gov/submissions/CIK{cik}.json"
    j=fetch_json(sub_url)
    def pack(rec): 
        return pd.DataFrame({
            "accession":rec["accessionNumber"],
            "form":rec["form"],
            "filed":rec["filingDate"],
            "primary":rec["primaryDocument"],
            "source":"json"})
    recent_df=pack(j["filings"]["recent"])
    older=[]
    for f in j.get("filings",{}).get("files",[]):
        y=fetch_json(f"https://data.sec.gov/submissions/{f['name']}")
        if "filings" in y and "recent" in y["filings"]:
            older.append(pack(y["filings"]["recent"]))
        time.sleep(PAUSE)
    all_filings=pd.concat([recent_df]+older,ignore_index=True)
    all_filings["filed"]=pd.to_datetime(all_filings["filed"])
    return all_filings

def get_filings_atom(cik,form="10-K"):
    url=f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type={form}&owner=exclude&count=200&output=atom"
    soup=BeautifulSoup(fetch_text(url),"xml")
    entries=[]
    for e in soup.find_all("entry"):
        filed=e.find("filing-date").text if e.find("filing-date") else None
        link=e.find("link")["href"] if e.find("link") else None
        acc=e.find("accession-number").text if e.find("accession-number") else None
        entries.append((acc,form,filed,link))
    return entries

def merge_sources(cik):
    j=get_filings_json(cik)
    a=[]
    for f in FORMS:
        a+=get_filings_atom(cik,f); time.sleep(PAUSE)
    df=pd.concat([j,pd.DataFrame(a,columns=["accession","form","filed","primary"])],ignore_index=True)
    df["filed"]=pd.to_datetime(df["filed"])
    df=df[df["form"].isin(FORMS)]
    def to_abs(u):
        u=str(u or "")
        return u if u.startswith("http") else urljoin("https://www.sec.gov/Archives/",u)
    df["primary"]=df["primary"].map(to_abs)
    return df.drop_duplicates(subset=["accession","form","filed"]).sort_values("filed")

# ------------------- URL resolver (index -> main document) -------------------
def resolve_document_url(index_or_doc_url: str):
    """
    Given a 'primary' URL that may be an -index.htm page, return the actual 10-K/10-Q HTML doc URL.
    Strategy:
      1) If URL is already an .htm/.html and not *-index.htm*, keep it.
      2) Else load the index page, collect all .htm/.html links under /Archives/,
         prefer ones containing 10-k/10-q/form10k/form10q, otherwise pick the first HTML doc.
    """
    u = str(index_or_doc_url or "")
    low = u.lower()
    if low.endswith((".htm",".html")) and "-index" not in low:
        return u

    # fetch index page and pick document
    try:
        html = fetch_text(u)
    except Exception:
        return u

    soup = BeautifulSoup(html, "lxml")
    hrefs = [a.get("href") for a in soup.find_all("a") if a.get("href")]
    hrefs = [urljoin(u, h) for h in hrefs if h.lower().endswith((".htm",".html"))]

    # rank candidates
    def rank(x):
        xl = x.lower()
        score = 100
        if "-index" in xl: score += 50
        if "10-k" in xl or "form10k" in xl: score -= 20
        if "10-q" in xl or "form10q" in xl: score -= 20
        # prefer docs in /Archives/edgar/data/
        if "/edgar/data/" in xl: score -= 10
        return score

    cands = sorted(set(hrefs), key=rank)
    for c in cands:
        if "-index" not in c.lower():
            return c
    return cands[0] if cands else u

# ------------------- Helpers -------------------
def norm(s):
    if s is None: return ""
    s=str(s).replace("\u00A0"," ")
    return re.sub(r"\s+"," ",s).strip()

def parse_number(s):
    if not s: return None
    s=str(s).strip()
    neg = s.startswith("(") and s.endswith(")")
    num = re.sub(r"[^\d\.\-]","",s)
    if num in ("","-","."): return None
    try:
        v=float(num)
        return -v if neg else v
    except:
        return None

def detect_scale(html, headers_joined):
    t=(html or "").lower()+" "+(headers_joined or "").lower()
    if "in millions" in t or "(millions)" in t: return 1_000_000.0
    if "in thousands" in t or "(thousands)" in t: return 1_000.0
    return 1.0

def try_parse_date(s):
    s = norm(s)
    s = re.sub(r"(?i)\b(as of|for the (three|six|nine|twelve)\s+months\s+ended|for the (quarter|year)\s+ended|ended)\b", "", s).strip(",;: ")
    for fmt in DATE_PATTERNS:
        try: return datetime.strptime(s, fmt).date()
        except Exception: pass
    m = re.search(r"([A-Za-z]{3,9}\s+\d{1,2},\s+\d{4})", s)
    if m:
        for fmt in ("%B %d, %Y","%b %d, %Y"):
            try: return datetime.strptime(m.group(1), fmt).date()
            except: pass
    return None

def header_periods(df):
    out=[]
    for c in df.columns:
        cstr=norm(c)
        m=YEAR_RE.search(cstr)
        if m: out.append((c,int(m.group()),"year")); continue
        d=try_parse_date(cstr)
        if d: out.append((c,d,"date"))
    if not out and not df.empty:
        head=[norm(x) for x in df.iloc[0].tolist()]
        tmp=[]
        for h in head:
            m=YEAR_RE.search(h)
            if m: tmp.append((h,int(m.group()),"year"))
            else:
                d=try_parse_date(h)
                if d: tmp.append((h,d,"date"))
        if tmp:
            df.columns=head
            df.drop(df.index[0], inplace=True)
            return tmp
    return out

def label_row_map(df):
    m={}
    firstcol = df.columns[0]
    # ensure string/object dtype before mapping to avoid dtype warnings
    df[firstcol] = df[firstcol].astype(object)
    for i, lbl in enumerate(df[firstcol].map(norm)):
        for key, pats in LABELS_HTML.items():
            if key in m: continue
            if any(re.fullmatch(p, lbl, flags=re.I) for p in pats):
                m[key]=i
    return m

def map_date_to_fyq(d: date):
    if d is None: return None, None, None
    m = d.month
    fy = d.year if m<=9 else d.year+1
    if   m in (10,11,12): fq = 1
    elif m in (1,2,3):    fq = 2
    elif m in (4,5,6):    fq = 3
    else:                 fq = 4
    return fy, fq, "Q"

def fy_end_date_from_year(y: int) -> date:
    try: return date(int(y), 9, 30)
    except: return None

# ------------------- Extractors -------------------
def extract_html_per_period(url):
    """Return list of (period_end, period_type, fy, fq, row_metrics_dict, src_url)."""
    doc_url = resolve_document_url(url)  # <-- ensure main doc
    html=fetch_text(doc_url)
    out=[]
    # Wrap with StringIO to avoid FutureWarning
    try:
        tables=pd.read_html(StringIO(html), flavor="lxml")
    except Exception:
        tables=[]
    if not tables:
        return out

    for t in tables:
        if t.empty or t.shape[1] < 2: 
            continue
        df=t.copy()
        df.columns=[norm(c) for c in df.columns]
        # Coerce first column to object before normalization (prevents dtype warnings)
        firstcol = df.columns[0]
        df[firstcol] = df[firstcol].astype(object)
        df[firstcol] = df[firstcol].map(norm)

        pcols=header_periods(df)
        if not pcols: 
            continue
        row_idx=label_row_map(df)
        if not row_idx:
            continue
        scale=detect_scale(html, " ".join(map(str,df.columns)))

        for col_label, per, ptype in pcols:
            metrics={}
            for k, ridx in row_idx.items():
                val=parse_number(df.iloc[ridx][col_label])
                if val is not None:
                    metrics[k]=val*scale
            if not metrics:
                continue

            if ptype=="date":
                fy,fq,kind = map_date_to_fyq(per)
                period_end = per
                period_type = f"{kind}{fq}" if fq else "FY"
            else:  # year-only column -> assume annual
                fy = int(per)
                fq = None
                period_type = "FY"
                period_end = fy_end_date_from_year(fy)

            mapped={
                "revenue": metrics.get("revenue"),
                "net_income": metrics.get("net_income"),
                "total_assets": metrics.get("assets"),
                "total_liabilities": metrics.get("liabilities"),
                "shareholders_equity": metrics.get("equity"),
            }
            out.append((period_end, period_type, fy, fq, mapped, doc_url))
    return out

def extract_text_metrics(html_or_text):
    text=(html_or_text or "")
    metrics={}
    for key,pats in LABELS_TEXT.items():
        for pat in pats:
            m=re.search(pat, text, flags=re.I)
            if m:
                full = m.group(0)
                neg = "(" in full and ")" in full
                val=parse_number(m.group(1))
                if val is not None and neg: val = -abs(val)
                metrics[key]=val
                break
    return metrics

# ------------------- MAIN -------------------
def main():
    filings=merge_sources(CIK)
    filings=filings[(filings["filed"]>=DATE_FROM)&(filings["filed"]<=DATE_TO)].copy()

    period_rows = {}  # key=(period_type, fy, fq)

    for _,r in filings.iterrows():
        url=r["primary"]
        filed=r["filed"].strftime("%Y-%m-%d")
        form=r["form"]

        try:
            per_list=extract_html_per_period(url)
            if per_list:
                got=0
                for period_end, period_type, fy, fq, row, src in per_list:
                    if fy is None: 
                        continue
                    key=(period_type, fy, fq)
                    rec=period_rows.setdefault(key, {
                        "period_end": period_end.isoformat() if isinstance(period_end, date) else None,
                        "period_type": period_type, "fiscal_year": fy, "fiscal_quarter": fq,
                        "filing_date": filed, "form": form, "source_url": src,
                        "revenue": None,"net_income":None,"total_assets":None,
                        "total_liabilities":None,"shareholders_equity":None
                    })
                    for k_in, k_out in [
                        ("revenue","revenue"), ("net_income","net_income"),
                        ("total_assets","total_assets"), ("total_liabilities","total_liabilities"),
                        ("shareholders_equity","shareholders_equity")
                    ]:
                        v=row.get(k_in)
                        if v is not None and rec[k_out] is None:
                            rec[k_out]=v; got+=1
                if got:
                    print(f"[HTML] {filed} {form} -> captured {got} values")
                    time.sleep(PAUSE)
                    continue
        except Exception as e:
            print(f"[HTML-ERR] {filed} {form}: {e}")

        # Last-resort text fallback mapped to filing period
        try:
            doc_url = resolve_document_url(url)
            h=fetch_text(doc_url)
            m=extract_text_metrics(h)
            if m:
                fdate = pd.to_datetime(r["filed"]).date()
                fy,fq,_ = map_date_to_fyq(fdate)
                if form == "10-K": 
                    fq=None; period_type="FY"; pend = fy_end_date_from_year(fy)
                else:
                    period_type=f"Q{fq}"; pend = fdate
                key=(period_type, fy, fq)
                rec=period_rows.setdefault(key, {
                    "period_end": pend.isoformat() if isinstance(pend, date) else None,
                    "period_type": period_type, "fiscal_year": fy, "fiscal_quarter": fq,
                    "filing_date": filed, "form": form, "source_url": doc_url,
                    "revenue": None,"net_income":None,"total_assets":None,
                    "total_liabilities":None,"shareholders_equity":None
                })
                if m.get("revenue") is not None and rec["revenue"] is None: rec["revenue"]=m["revenue"]
                if m.get("net_income") is not None and rec["net_income"] is None: rec["net_income"]=m["net_income"]
                if m.get("assets") is not None and rec["total_assets"] is None: rec["total_assets"]=m["assets"]
                if m.get("liabilities") is not None and rec["total_liabilities"] is None: rec["total_liabilities"]=m["liabilities"]
                if m.get("equity") is not None and rec["shareholders_equity"] is None: rec["shareholders_equity"]=m["equity"]
                print(f"[TEXT] {filed} {form} -> captured some values")
        except Exception as e:
            print(f"[TEXT-ERR] {filed} {form}: {e}")

        time.sleep(PAUSE)

    # --- BUILD OUTPUT DATAFRAME SAFELY ---
    cols = ["period_end","period_type","fiscal_year","fiscal_quarter",
            "filing_date","form","revenue","net_income","total_assets",
            "total_liabilities","shareholders_equity","source_url"]
    if not period_rows:
        print("[WARN] No metrics captured. Dumping filings seen to help debug:")
        try:
            print(filings[["filed","form","primary"]].to_string(index=False))
        except Exception:
            pass
        out=pd.DataFrame(columns=cols)
    else:
        out=pd.DataFrame([
            {
                "period_end": v.get("period_end"),
                "period_type": v.get("period_type"),
                "fiscal_year": v.get("fiscal_year"),
                "fiscal_quarter": v.get("fiscal_quarter"),
                "filing_date": v.get("filing_date"),
                "form": v.get("form"),
                "revenue": v.get("revenue"),
                "net_income": v.get("net_income"),
                "total_assets": v.get("total_assets"),
                "total_liabilities": v.get("total_liabilities"),
                "shareholders_equity": v.get("shareholders_equity"),
                "source_url": v.get("source_url"),
            }
            for _, v in sorted(period_rows.items(), key=lambda kv: (kv[0][1], kv[0][0], kv[0][2] or 0))
        ])

    # Filter by filing date and write
    if not out.empty and "filing_date" in out.columns:
        out["filing_date"]=pd.to_datetime(out["filing_date"])
        out=out[(out["filing_date"]>=DATE_FROM)&(out["filing_date"]<=DATE_TO)]
        out=out.sort_values(["fiscal_year","period_type"])

    out.to_csv("apple_financials_2001_2009_periods.csv",index=False)
    print(f"[DONE] Saved {len(out)} rows to apple_financials_2001_2009_periods.csv")

if __name__=="__main__":
    main()


[HTML] 2001-05-14 10-Q -> captured 3 values
[TEXT] 2001-08-13 10-Q -> captured some values
[HTML] 2001-12-21 10-K -> captured 8 values
[HTML] 2002-02-11 10-Q -> captured 1 values
[TEXT] 2002-05-14 10-Q -> captured some values
[TEXT] 2002-08-09 10-Q -> captured some values
[HTML] 2002-12-19 10-K -> captured 6 values
[TEXT] 2003-02-10 10-Q -> captured some values
[TEXT] 2003-05-13 10-Q -> captured some values
[HTML] 2003-12-19 10-K -> captured 4 values
[HTML] 2004-12-03 10-K -> captured 6 values
[TEXT] 2006-05-05 10-Q -> captured some values
[TEXT] 2006-12-29 10-Q -> captured some values
[HTML] 2007-11-15 10-K -> captured 12 values
[TEXT] 2008-11-05 10-K -> captured some values
[TEXT] 2009-04-23 10-Q -> captured some values
[DONE] Saved 21 rows to apple_financials_2001_2009_periods.csv
